# Telco Customer Churn: Mini End-to-End Data Science Project

**Problem:** a telecom company loses customers every month ("churn"). Can we understand *who* churns and *why*, and build a model that flags customers at risk so the retention team can act before they leave?

**Workflow in this notebook**
1. Data collection
2. Data understanding and quality checks
3. Data cleaning (saves the final cleaned dataset)
4. Exploratory data analysis (EDA)
5. Feature engineering
6. Modeling and model selection (cross-validation)
7. Evaluation on a held-out test set
8. Interpretation, insights and recommendations (writes the project report)

**Dataset:** IBM's *Telco Customer Churn* sample dataset (fictional telecom customers): one row per customer, 21 columns covering demographics, services, account details and whether the customer left (`Churn`).

**Run it top to bottom.** The numbers in the report `docs/project_report.md` are generated by the last cell from this run.

## 1. Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_predict,
                                     cross_validate, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
RED, BLUE = "#d6432f", "#2a7fbf"          # red = churned, blue = stayed (used in every chart)

# Works inside the repo (notebooks/ folder) and in Google Colab
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
RAW_DIR, PROC_DIR = ROOT / "data" / "raw", ROOT / "data" / "processed"
IMG_DIR, DOCS_DIR = ROOT / "images", ROOT / "docs"
for d in (RAW_DIR, PROC_DIR, IMG_DIR, DOCS_DIR):
    d.mkdir(parents=True, exist_ok=True)


def save_fig(name):
    """Save the current figure to images/ and show it."""
    plt.tight_layout()
    plt.savefig(IMG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()


R = {}   # key numbers collected along the way; used to write the report at the end

## 2. Data collection
The notebook looks for the raw CSV in the repo (`data/raw/`). If it is not there, it downloads the IBM copy and saves it, so the raw data is always kept unchanged in `data/raw/`.

In [ ]:
RAW_NAME = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
candidates = [RAW_DIR / RAW_NAME, ROOT / "data" / RAW_NAME, ROOT / RAW_NAME, RAW_DIR / "Telco-Customer-Churn.csv"]
raw_path = next((p for p in candidates if p.exists()), None)

if raw_path is None:
    try:
        pd.read_csv(URL).to_csv(RAW_DIR / RAW_NAME, index=False)
        raw_path = RAW_DIR / RAW_NAME
        print("Downloaded the dataset to", raw_path)
    except Exception as exc:
        raise FileNotFoundError(
            f"Could not find or download the dataset ({exc}). Download the Telco churn CSV manually and save it as "
            f"{RAW_DIR / RAW_NAME}") from exc

raw = pd.read_csv(raw_path)
EXPECTED = ["customerID", "gender", "SeniorCitizen", "Partner", "Dependents", "tenure", "PhoneService",
            "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
            "TechSupport", "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
            "MonthlyCharges", "TotalCharges", "Churn"]
missing_cols = [c for c in EXPECTED if c not in raw.columns]
assert not missing_cols, f"Unexpected file: missing columns {missing_cols}"
print("Loaded", raw_path, "->", raw.shape)
raw.head()

## 3. Data understanding and quality checks
Look for missing values (including hidden ones such as blank strings), duplicates, wrong data types and inconsistent categories before changing anything.

In [ ]:
raw.info()

In [ ]:
raw.describe().T

In [ ]:
blank_total = int(raw["TotalCharges"].isna().sum() + raw["TotalCharges"].astype(str).str.strip().eq("").sum())
dup_rows = int(raw.duplicated().sum())
dup_ids = int(raw["customerID"].duplicated().sum())
print("Rows, columns:", raw.shape)
print("Obvious missing values (NaN):", int(raw.isna().sum().sum()))
print("Blank or missing TotalCharges (hidden by the text type):", blank_total)
print("Duplicate rows:", dup_rows, "| duplicate customerIDs:", dup_ids)
print("TotalCharges data type:", raw["TotalCharges"].dtype, "(should be numeric)")
print("tenure of the customers with blank TotalCharges:",
      sorted(raw.loc[pd.to_numeric(raw["TotalCharges"], errors="coerce").isna(), "tenure"].unique()))
print()
for col in ["SeniorCitizen", "MultipleLines", "InternetService", "OnlineSecurity", "Contract", "PaymentMethod", "Churn"]:
    print(raw[col].value_counts(dropna=False).to_string(), "\n")

R.update(n_rows=len(raw), n_cols=raw.shape[1], blank_total=blank_total, dup_rows=dup_rows, dup_ids=dup_ids)

Typical issues in this dataset (confirm them in the output above):
- `TotalCharges` is stored as text because some values are blank. Those blanks are real missing values.
- Add-on columns use three values ("Yes", "No", "No internet service") where "No internet service" repeats what `InternetService` already says. `MultipleLines` has the same issue with "No phone service".
- `SeniorCitizen` is coded 0/1 while every other yes/no column uses "Yes"/"No".

## 4. Data cleaning
The raw data is never modified. Every step is recorded in a cleaning log.

In [ ]:
df = raw.copy()
log = []
ADDONS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

# 1. Remove stray spaces in text columns
text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
for c in text_cols:
    df[c] = df[c].str.strip()
log.append(("Stripped leading/trailing spaces from text columns", len(text_cols)))

# 2. TotalCharges: text -> number (blanks become NaN)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
n_nan = int(df["TotalCharges"].isna().sum())
log.append(("Converted TotalCharges from text to number (blanks became missing)", n_nan))

# 3. Fill the missing TotalCharges: a customer with tenure 0 has not been billed yet, so the total is 0
new_customers = df["TotalCharges"].isna() & (df["tenure"] == 0)
df.loc[new_customers, "TotalCharges"] = 0.0
rest = df["TotalCharges"].isna()
df.loc[rest, "TotalCharges"] = df.loc[rest, "MonthlyCharges"] * df.loc[rest, "tenure"]   # safety net
log.append(("Filled missing TotalCharges (0 for customers with tenure 0)", n_nan))

# 4. SeniorCitizen 0/1 -> No/Yes, to match the other yes/no columns
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
log.append(("Recoded SeniorCitizen from 0/1 to No/Yes", len(df)))

# 5. Collapse redundant categories
n_collapsed = int((df[ADDONS] == "No internet service").sum().sum() + (df["MultipleLines"] == "No phone service").sum())
df[ADDONS] = df[ADDONS].replace("No internet service", "No")
df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")
log.append(("Replaced 'No internet service' / 'No phone service' with 'No' (already captured by InternetService and PhoneService)", n_collapsed))

# 6. Duplicates
before = len(df)
df = df.drop_duplicates(subset="customerID")
log.append(("Removed duplicate customers", before - len(df)))

pd.DataFrame(log, columns=["Cleaning step", "Values / rows affected"])

In [ ]:
# Validation: fail loudly if the cleaned data is not what we expect
assert df.isna().sum().sum() == 0, "missing values remain"
assert df["customerID"].is_unique
assert (df["tenure"] >= 0).all() and (df["MonthlyCharges"] > 0).all() and (df["TotalCharges"] >= 0).all()
assert set(df["Churn"].unique()) == {"Yes", "No"}
print("All checks passed. Shape after cleaning:", df.shape)

# Outliers: flagged with the IQR rule, kept because they are genuine prices
for col in ["MonthlyCharges", "TotalCharges"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    n_out = int(((df[col] < q1 - 1.5 * (q3 - q1)) | (df[col] > q3 + 1.5 * (q3 - q1))).sum())
    print(f"{col}: {n_out} IQR outliers (kept)")

In [ ]:
# Save the FINAL CLEANED DATASET
CLEAN_PATH = PROC_DIR / "telco_churn_cleaned.csv"
df.to_csv(CLEAN_PATH, index=False)
R.update(cleaning_log=log, n_clean=len(df))
print("Saved", CLEAN_PATH, df.shape)
df.head()

## 5. Exploratory data analysis (EDA)
We add a numeric churn flag and a few helper columns, then look at how churn differs between groups.

In [ ]:
def add_features(d):
    """Extra columns derived from the cleaned data."""
    d = d.copy()
    d["ChurnFlag"] = (d["Churn"] == "Yes").astype(int)
    d["NumAddOns"] = (d[ADDONS] == "Yes").sum(axis=1)                     # how many add-on services
    d["AutoPay"] = np.where(d["PaymentMethod"].str.contains("automatic", case=False), "Yes", "No")
    d["HasInternet"] = np.where(d["InternetService"] != "No", "Yes", "No")
    d["TenureGroup"] = pd.cut(d["tenure"], bins=[-1, 12, 24, 48, np.inf],
                              labels=["0-12 months", "13-24 months", "25-48 months", "49+ months"])
    return d


df = add_features(df)
overall = df["ChurnFlag"].mean()
R.update(overall=overall, n_churn=int(df["ChurnFlag"].sum()))


def present(order, col):
    return [v for v in order if v in set(df[col].astype(str))]


def plot_rate(ax, col, title, order=None):
    """Bar chart of the churn rate for each group of `col`, with the overall rate as a dashed line."""
    s = df.groupby(col, observed=True)["ChurnFlag"].agg(["mean", "count"])
    if order:
        s = s.reindex(order)
    vals = s["mean"] * 100
    bars = ax.bar(s.index.astype(str), vals, color=RED)
    ax.bar_label(bars, labels=[f"{v:.1f}%" for v in vals], padding=2)
    ax.axhline(overall * 100, color="black", linestyle="--", linewidth=1, label=f"Overall churn {overall:.1%}")
    ax.set_ylim(0, vals.max() * 1.25)
    ax.set_xlabel(col)
    ax.set_ylabel("Churn rate (%)")
    ax.set_title(title)
    ax.legend(loc="upper right")
    return s


print(f"Overall churn rate: {overall:.1%} ({R['n_churn']} of {len(df)} customers)")

fig, ax = plt.subplots(figsize=(5.5, 4.5))
counts = df["Churn"].value_counts().reindex(["No", "Yes"])
bars = ax.bar(["Stayed", "Churned"], counts.values, color=[BLUE, RED])
ax.bar_label(bars, labels=[f"{n} ({n / len(df):.1%})" for n in counts.values], padding=3)
ax.set_ylim(0, counts.max() * 1.15)
ax.set_ylabel("Number of customers")
ax.set_xlabel("Outcome")
ax.set_title(f"{overall:.0%} of customers churned")
save_fig("01_churn_distribution")

### 5.1 Contract type and tenure

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
R["contract"] = plot_rate(ax[0], "Contract", "Churn rate by contract type", present(["Month-to-month", "One year", "Two year"], "Contract"))
R["tenure_group"] = plot_rate(ax[1], "TenureGroup", "Churn rate by customer tenure", list(df["TenureGroup"].cat.categories))
save_fig("02_churn_by_contract_and_tenure")
R["contract"].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
bins = np.linspace(0, df["tenure"].max(), 37)
ax.hist(df.loc[df["Churn"] == "No", "tenure"], bins=bins, alpha=0.6, color=BLUE, density=True, label="Stayed")
ax.hist(df.loc[df["Churn"] == "Yes", "tenure"], bins=bins, alpha=0.6, color=RED, density=True, label="Churned")
ax.set_xlabel("Tenure (months with the company)")
ax.set_ylabel("Share of customers (density)")
ax.set_title("Tenure distribution: churned vs stayed")
ax.legend(title="Outcome")
save_fig("03_tenure_distribution")

### 5.2 Prices and services

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
data_box = [df.loc[df["Churn"] == "No", "MonthlyCharges"], df.loc[df["Churn"] == "Yes", "MonthlyCharges"]]
bp = ax.boxplot(data_box, patch_artist=True, widths=0.5)
for patch, colour in zip(bp["boxes"], [BLUE, RED]):
    patch.set_facecolor(colour)
    patch.set_alpha(0.6)
ax.set_xticks([1, 2])
ax.set_xticklabels(["Stayed", "Churned"])
ax.set_xlabel("Outcome")
ax.set_ylabel("Monthly charges")
ax.set_title("Monthly charges: churned vs stayed")
save_fig("04_monthly_charges_boxplot")
R["mc_mean"] = df.groupby("Churn")["MonthlyCharges"].mean()
R["mc_mean"].round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
R["internet"] = plot_rate(ax[0], "InternetService", "Churn rate by internet service", present(["DSL", "Fiber optic", "No"], "InternetService"))
pay_order = list(df.groupby("PaymentMethod")["ChurnFlag"].mean().sort_values(ascending=False).index)
R["payment"] = plot_rate(ax[1], "PaymentMethod", "Churn rate by payment method", pay_order)
plt.setp(ax[1].get_xticklabels(), rotation=20, ha="right")
save_fig("05_churn_by_internet_and_payment")

In [ ]:
net = df[df["HasInternet"] == "Yes"].copy()
net["AddOnGroup"] = pd.cut(net["NumAddOns"], bins=[-1, 0, 2, 6], labels=["0 add-ons", "1-2 add-ons", "3+ add-ons"])
fig, ax = plt.subplots(figsize=(7, 4.8))
s = net.groupby("AddOnGroup", observed=True)["ChurnFlag"].mean() * 100
bars = ax.bar(s.index.astype(str), s.values, color=RED)
ax.bar_label(bars, labels=[f"{v:.1f}%" for v in s.values], padding=2)
ax.set_ylim(0, s.max() * 1.25)
ax.set_xlabel("Add-on services (security, backup, protection, support, streaming)")
ax.set_ylabel("Churn rate (%)")
ax.set_title("Churn rate by number of add-on services (internet customers)")
save_fig("06_churn_by_addons")
R["addons"] = s

### 5.3 Other customer characteristics

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, (col, title) in zip(ax.flat, [("SeniorCitizen", "Senior citizen"), ("Partner", "Has a partner"),
                                      ("Dependents", "Has dependents"), ("PaperlessBilling", "Paperless billing"),
                                      ("gender", "Gender"), ("AutoPay", "Pays automatically")]):
    plot_rate(a, col, f"Churn rate: {title}", present(["No", "Yes", "Female", "Male"], col))
save_fig("07_churn_by_customer_traits")
R["traits"] = {c: df.groupby(c)["ChurnFlag"].mean() for c in ["SeniorCitizen", "Partner", "Dependents", "PaperlessBilling", "gender", "AutoPay"]}

In [ ]:
df["SeniorFlag"] = (df["SeniorCitizen"] == "Yes").astype(int)
num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "NumAddOns", "SeniorFlag", "ChurnFlag"]
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=40, ha="right")
ax.set_yticks(range(len(num_cols)))
ax.set_yticklabels(num_cols)
for (i, j), v in np.ndenumerate(corr.values):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, label="Correlation")
ax.set_title("Correlation between numeric variables")
save_fig("08_correlation_heatmap")
R["corr_churn"] = corr["ChurnFlag"].drop("ChurnFlag").round(3)
R["corr_churn"]

## 6. Feature engineering and preprocessing
Features added during EDA: `NumAddOns` (count of add-on services), `AutoPay` (pays by bank transfer or credit card automatically) and `HasInternet`.
- Numeric columns are standardised (`StandardScaler`).
- Text categories are one-hot encoded (`OneHotEncoder`).
- Both steps are part of the model **pipeline**, so they are fitted on training data only. This avoids data leakage.
- `customerID` is dropped (an identifier) and `TenureGroup` is not used because `tenure` already carries the information.

In [ ]:
y = df["ChurnFlag"]
X = df.drop(columns=["customerID", "Churn", "ChurnFlag", "TenureGroup", "SeniorFlag"])
NUM_COLS = ["tenure", "MonthlyCharges", "TotalCharges", "NumAddOns"]
CAT_COLS = [c for c in X.columns if c not in NUM_COLS]
print("Numeric features:", NUM_COLS)
print("Categorical features:", CAT_COLS)


def make_pipeline(model):
    prep = ColumnTransformer([("num", StandardScaler(), NUM_COLS),
                              ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), CAT_COLS)])
    return Pipeline([("prep", prep), ("clf", model)])


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(f"Train: {X_train.shape}, Test: {X_test.shape}, churn rate train/test: {y_train.mean():.1%} / {y_test.mean():.1%}")
R.update(n_train=len(X_train), n_test=len(X_test), n_features_raw=X.shape[1])

## 7. Modeling and model selection
Four models plus a "do nothing" baseline are compared with **stratified 5-fold cross-validation on the training set only**. The test set stays untouched until the very end.

Churn is the minority class, so accuracy alone is misleading. We rank models by **ROC AUC** (how well they rank churners above non-churners) and also report precision, recall and F1 for the churn class.

In [ ]:
models = {
    "Dummy (most frequent)": make_pipeline(DummyClassifier(strategy="most_frequent")),
    "Logistic Regression": make_pipeline(LogisticRegression(max_iter=2000)),
    "Decision Tree": make_pipeline(DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)),
    "Random Forest": make_pipeline(RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)),
    "Gradient Boosting": make_pipeline(GradientBoostingClassifier(random_state=RANDOM_STATE)),
}
COMPLEXITY = {"Dummy (most frequent)": 0, "Logistic Regression": 1, "Decision Tree": 2, "Random Forest": 3, "Gradient Boosting": 4}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"roc_auc": "roc_auc", "accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1"}

rows, fold_auc = [], {}
for name, model in models.items():
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    fold_auc[name] = res["test_roc_auc"]
    rows.append({"Model": name, "ROC AUC": res["test_roc_auc"].mean(), "ROC AUC std": res["test_roc_auc"].std(),
                 "Accuracy": res["test_accuracy"].mean(), "Precision": res["test_precision"].mean(),
                 "Recall": res["test_recall"].mean(), "F1": res["test_f1"].mean()})
cv_table = pd.DataFrame(rows).sort_values("ROC AUC", ascending=False).reset_index(drop=True)
R["cv_table"] = cv_table
cv_table.round(3)

In [ ]:
order = list(cv_table["Model"])
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(order[::-1], [cv_table.set_index("Model").loc[m, "ROC AUC"] for m in order[::-1]],
        xerr=[cv_table.set_index("Model").loc[m, "ROC AUC std"] for m in order[::-1]], color=BLUE, capsize=3)
ax.set_xlabel("Cross-validated ROC AUC (mean +/- standard deviation over 5 folds)")
ax.set_ylabel("Model")
ax.set_title("Model comparison on the training set")
ax.set_xlim(0.4, 1.0)
save_fig("09_model_comparison")

**Choosing the model.** The best model by mean ROC AUC is the starting point. Any model within one standard deviation of it is treated as equally good, and then the **simplest** of those is chosen, because a simpler model is easier to explain to the business and less likely to overfit.

In [ ]:
real = cv_table[cv_table["Model"] != "Dummy (most frequent)"]
best_row = real.iloc[0]
close = real[real["ROC AUC"] >= best_row["ROC AUC"] - best_row["ROC AUC std"]]["Model"].tolist()
chosen = min(close, key=lambda m: COMPLEXITY[m])
print("Best by mean ROC AUC:", best_row["Model"])
print("Within one standard deviation:", close)
print("Chosen (simplest of these):", chosen)

GRIDS = {
    "Logistic Regression": {"clf__C": [0.01, 0.1, 1, 10]},
    "Decision Tree": {"clf__max_depth": [3, 4, 5, 6, 8], "clf__min_samples_leaf": [5, 20, 50]},
    "Random Forest": {"clf__max_depth": [None, 8, 12], "clf__min_samples_leaf": [3, 5, 10]},
    "Gradient Boosting": {"clf__n_estimators": [100, 200], "clf__learning_rate": [0.05, 0.1], "clf__max_depth": [2, 3]},
}
gs = GridSearchCV(models[chosen], GRIDS[chosen], scoring="roc_auc", cv=cv, n_jobs=-1).fit(X_train, y_train)
final_model = gs.best_estimator_
best_params = {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}
print("Best parameters:", best_params, "| tuned CV ROC AUC:", round(gs.best_score_, 4))
R.update(chosen=chosen, close=close, best_by_auc=best_row["Model"], best_params=best_params, tuned_auc=gs.best_score_)

### Choosing the decision threshold
A model outputs a churn *probability*. The default rule "flag if probability is 0.5 or more" is rarely best for an imbalanced problem. We pick the threshold that gives the best F1 on **out-of-fold predictions from the training data**, never on the test set.

In [ ]:
oof = cross_val_predict(clone(final_model), X_train, y_train, cv=cv, method="predict_proba")[:, 1]
thresholds = np.arange(0.10, 0.91, 0.05)
thr_rows = [{"Threshold": t, "Precision": precision_score(y_train, oof >= t, zero_division=0),
             "Recall": recall_score(y_train, oof >= t), "F1": f1_score(y_train, oof >= t)} for t in thresholds]
thr_table = pd.DataFrame(thr_rows)
best_thr = float(thr_table.loc[thr_table["F1"].idxmax(), "Threshold"])
print(f"Best threshold on training data (max F1): {best_thr:.2f}")

fig, ax = plt.subplots(figsize=(8, 4.8))
for col, colour in [("Precision", BLUE), ("Recall", RED), ("F1", "#555555")]:
    ax.plot(thr_table["Threshold"], thr_table[col], marker="o", color=colour, label=col)
ax.axvline(best_thr, color="black", linestyle="--", linewidth=1, label=f"Chosen threshold {best_thr:.2f}")
ax.set_xlabel("Decision threshold (flag customer as churn if probability is at least this)")
ax.set_ylabel("Score (out-of-fold, training data)")
ax.set_title("Precision/recall trade-off")
ax.legend()
save_fig("10_threshold_tradeoff")
R.update(best_thr=best_thr)
thr_table.round(3)

## 8. Evaluation on the held-out test set
The test set is used once, with the final model and the chosen threshold.

In [ ]:
proba = final_model.predict_proba(X_test)[:, 1]


def metrics_at(threshold):
    pred = (proba >= threshold).astype(int)
    return {"Accuracy": accuracy_score(y_test, pred), "Precision": precision_score(y_test, pred, zero_division=0),
            "Recall": recall_score(y_test, pred), "F1": f1_score(y_test, pred)}, confusion_matrix(y_test, pred)


m_default, cm_default = metrics_at(0.5)
m_tuned, cm_tuned = metrics_at(best_thr)
auc_test = roc_auc_score(y_test, proba)
test_table = pd.DataFrame({"Threshold 0.50": m_default, f"Threshold {best_thr:.2f}": m_tuned}).T
test_table["ROC AUC"] = auc_test
R.update(m_default=m_default, m_tuned=m_tuned, cm_default=cm_default, cm_tuned=cm_tuned, auc_test=auc_test)
print(f"Final model: {chosen} {best_params}")
test_table.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
for a, cm, title in [(ax[0], cm_default, "Threshold 0.50"), (ax[1], cm_tuned, f"Threshold {best_thr:.2f}")]:
    a.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        a.text(j, i, str(v), ha="center", va="center", fontsize=16, color="white" if v > cm.max() / 2 else "black")
    a.set_xticks([0, 1])
    a.set_xticklabels(["Stayed", "Churned"])
    a.set_yticks([0, 1])
    a.set_yticklabels(["Stayed", "Churned"])
    a.set_xlabel("Predicted")
    a.set_ylabel("Actual")
    a.set_title(f"Test confusion matrix, {title}")
save_fig("11_confusion_matrices")

fpr, tpr, _ = roc_curve(y_test, proba)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, color=RED, linewidth=2, label=f"{chosen} (AUC {auc_test:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random guessing")
ax.set_xlabel("False positive rate (loyal customers flagged)")
ax.set_ylabel("True positive rate (churners caught)")
ax.set_title("ROC curve on the test set")
ax.legend(loc="lower right")
save_fig("12_roc_curve")

## 9. Interpretation: what drives churn?
**Permutation importance:** shuffle one feature at a time on the test set and see how much the ROC AUC drops. A big drop means the model relies on that feature. It works for any model and reports the original features, not the one-hot columns.

In [ ]:
perm = permutation_importance(final_model, X_test, y_test, scoring="roc_auc", n_repeats=10,
                              random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
top = imp.head(10)[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top.index, top.values, color=RED)
ax.set_xlabel("Drop in test ROC AUC when the feature is shuffled")
ax.set_ylabel("Feature")
ax.set_title("Top 10 features the model relies on")
save_fig("13_permutation_importance")
R["imp"] = imp
imp.head(10).round(4)

## 10. Insights, recommendations and the project report
The next cell turns the results above into `docs/project_report.md` (problem, approach, results, insights), so every number in the report comes from this run.

In [ ]:
def pct(x):
    return f"{x:.1%}"


def md_table(frame, fmt="{:.3f}"):
    cols = list(frame.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in frame.iterrows():
        lines.append("| " + " | ".join(fmt.format(v) if isinstance(v, (float, np.floating)) else str(v) for v in r) + " |")
    return "\n".join(lines)


ct, tg, inet, pay, add = R["contract"], R["tenure_group"], R["internet"], R["payment"], R["addons"]
hi_c, lo_c = ct["mean"].idxmax(), ct["mean"].idxmin()
first_t, last_t = tg.index[0], tg.index[-1]
hi_i, lo_i = inet["mean"].idxmax(), inet["mean"].idxmin()
hi_p, lo_p = pay["mean"].idxmax(), pay["mean"].idxmin()
mc_yes, mc_no = R["mc_mean"].get("Yes", np.nan), R["mc_mean"].get("No", np.nan)
tr = R["traits"]
top3 = [f"`{f}`" for f in R["imp"].head(3).index]
tn, fp, fn, tp = R["cm_tuned"].ravel()
d_tn, d_fp, d_fn, d_tp = R["cm_default"].ravel()

cv_show = R["cv_table"].copy()
cv_show["ROC AUC"] = [f"{m:.3f} +/- {s:.3f}" for m, s in zip(cv_show["ROC AUC"], cv_show["ROC AUC std"])]
cv_show = cv_show.drop(columns="ROC AUC std")
test_show = test_table.round(3).reset_index().rename(columns={"index": "Decision threshold"})
log_tbl = pd.DataFrame(R["cleaning_log"], columns=["Cleaning step", "Values / rows affected"])
imp_tbl = R["imp"].head(10).round(4).rename("Drop in ROC AUC").reset_index().rename(columns={"index": "Feature"})

THRESH = 0.05      # a gap of at least 5 percentage points counts as a meaningful difference
ins = []
gap = ct.loc[hi_c, "mean"] - ct.loc[lo_c, "mean"]
if gap >= THRESH:
    ins.append(f"**Contract type is strongly linked to churn.** {hi_c} customers churn at {pct(ct.loc[hi_c, 'mean'])} versus {pct(ct.loc[lo_c, 'mean'])} for {lo_c} customers. Consider offering discounts or perks to move {hi_c} customers onto longer contracts.")
else:
    ins.append(f"**Contract type shows little difference in churn** (highest {pct(ct.loc[hi_c, 'mean'])}, lowest {pct(ct.loc[lo_c, 'mean'])}), so it is not a priority lever.")
gap = tg.loc[first_t, "mean"] - tg.loc[last_t, "mean"]
if gap >= THRESH:
    ins.append(f"**The first months are the riskiest.** Churn is {pct(tg.loc[first_t, 'mean'])} for customers with {first_t} of tenure versus {pct(tg.loc[last_t, 'mean'])} for {last_t}. Consider an onboarding programme and an early check-in for new customers.")
elif gap <= -THRESH:
    ins.append(f"**Churn rises with tenure.** It is {pct(tg.loc[last_t, 'mean'])} for {last_t} customers versus {pct(tg.loc[first_t, 'mean'])} for {first_t}. Long-standing customers may need loyalty offers.")
else:
    ins.append("**Tenure shows no large difference in churn** between new and long-standing customers.")
gap_i = inet.loc[hi_i, "mean"] - inet.loc[lo_i, "mean"]
price_note = (f" Churned customers also pay more per month on average ({mc_yes:.2f} vs {mc_no:.2f})." if mc_yes > mc_no else "")
if gap_i >= THRESH:
    ins.append(f"**Service type is linked to churn.** {hi_i} customers churn at {pct(inet.loc[hi_i, 'mean'])} versus {pct(inet.loc[lo_i, 'mean'])} for {lo_i}.{price_note} Review value for money and service quality in the {hi_i} segment before assuming price is the cause.")
else:
    ins.append(f"**Internet service type shows little difference in churn** (highest {pct(inet.loc[hi_i, 'mean'])}, lowest {pct(inet.loc[lo_i, 'mean'])}).")
a0, a3 = add.get("0 add-ons", np.nan) / 100, add.get("3+ add-ons", np.nan) / 100
if a0 - a3 >= THRESH:
    ins.append(f"**Add-on services go with loyalty.** Internet customers with no add-ons churn at {pct(a0)} versus {pct(a3)} with three or more. Bundling security, backup or support services may help retention, although the data cannot prove that add-ons cause loyalty.")
elif a3 - a0 >= THRESH:
    ins.append(f"**More add-ons go with higher churn** ({pct(a3)} with three or more vs {pct(a0)} with none), which is worth investigating, for example for price or service issues.")
else:
    ins.append("**Add-on services show no clear link to churn** in this data.")
gap = pay.loc[hi_p, "mean"] - pay.loc[lo_p, "mean"]
if gap >= THRESH:
    ins.append(f"**Payment method is a signal.** {hi_p} customers churn at {pct(pay.loc[hi_p, 'mean'])} versus {pct(pay.loc[lo_p, 'mean'])} for {lo_p}. Consider encouraging payment methods with lower churn.")
else:
    ins.append(f"**Payment method shows little difference in churn** (highest {pct(pay.loc[hi_p, 'mean'])}, lowest {pct(pay.loc[lo_p, 'mean'])}).")
prec = tp / (tp + fp) if tp + fp else 0
ins.append(f"**Use the model to prioritise outreach.** The model ranks customers by churn risk with a test ROC AUC of {R['auc_test']:.3f}. At the chosen threshold, contacting the flagged customers reaches {pct(tp / (tp + fn))} of the churners in the test set, and about {pct(prec)} of the people contacted would have churned. The retention team can raise or lower the threshold depending on how costly outreach is compared with losing a customer.")
ins.append(f"**Most influential features:** {', '.join(top3)} carry the most predictive weight in the model.")
insights_md = "\n".join(f"{i + 1}. {t}" for i, t in enumerate(ins))

report = f"""# Telco Customer Churn: Mini End-to-End Data Science Project

*This report is generated by `notebooks/end_to_end_churn_project.ipynb`; every number comes from that run.*

## 1. Problem
A telecom company loses customers every month. Keeping an existing customer is usually cheaper than winning a new one, so the retention team wants to (1) understand which customers churn and (2) flag customers at risk early enough to act.

**Goal:** analyse customer churn, build a model that predicts it, and turn the findings into practical recommendations.

## 2. Data
- **Source:** IBM *Telco Customer Churn* sample dataset (fictional telecom customers), also published on Kaggle.
- **Size:** {R['n_rows']:,} customers and {R['n_cols']} columns: demographics, services subscribed, contract and billing details, monthly and total charges, tenure, and the target `Churn` (Yes/No).
- **Churn rate:** {pct(R['overall'])} ({R['n_churn']:,} of {R['n_clean']:,} customers), so the classes are imbalanced.

## 3. Approach
### 3.1 Cleaning
Quality checks found {R['blank_total']} blank `TotalCharges` values hidden by a text data type, {R['dup_ids']} duplicate customer IDs and redundant categories. The raw file was kept unchanged and the following steps were applied:

{md_table(log_tbl, "{:.0f}")}

The final cleaned dataset ({R['n_clean']:,} rows) is saved as `data/processed/telco_churn_cleaned.csv`.

### 3.2 Exploratory data analysis
Churn rates were compared across contract type, tenure, internet service, payment method, add-on services and customer traits (see section 4.1).

### 3.3 Feature engineering and preprocessing
- New features: `NumAddOns` (count of add-on services), `AutoPay` (automatic payment) and `HasInternet`.
- Numeric features were standardised and text categories one-hot encoded inside a pipeline fitted on training data only, which prevents data leakage.

### 3.4 Modeling
- Stratified 80/20 train/test split ({R['n_train']:,} training, {R['n_test']:,} test customers, `random_state=42`).
- Four models and a "predict the most common class" baseline were compared with stratified 5-fold cross-validation on the training set.
- Models were ranked by ROC AUC. Any model within one standard deviation of the best was treated as equally good and the simplest of them was chosen.
- The chosen model was tuned with grid search, and the decision threshold was chosen on out-of-fold training predictions (maximum F1), never on the test set.

## 4. Results
### 4.1 What the data shows
- **Contract:** customers on **{hi_c}** contracts churn at {pct(ct.loc[hi_c, 'mean'])}, compared with {pct(ct.loc[lo_c, 'mean'])} on **{lo_c}** contracts.
- **Tenure:** churn is {pct(tg.loc[first_t, 'mean'])} for customers in their first period ({first_t}) and {pct(tg.loc[last_t, 'mean'])} for those with {last_t}.
- **Internet service:** the highest churn is among **{hi_i}** customers ({pct(inet.loc[hi_i, 'mean'])}) and the lowest among **{lo_i}** ({pct(inet.loc[lo_i, 'mean'])}).
- **Payment method:** **{hi_p}** has the highest churn ({pct(pay.loc[hi_p, 'mean'])}); **{lo_p}** has the lowest ({pct(pay.loc[lo_p, 'mean'])}).
- **Prices:** churned customers pay {mc_yes:.2f} per month on average, versus {mc_no:.2f} for customers who stayed.
- **Add-ons (internet customers):** {', '.join(f"{pct(v / 100)} churn with {k}" for k, v in add.items())}.
- **Customer traits:** churn is {pct(tr['SeniorCitizen'].get('Yes', np.nan))} for senior citizens vs {pct(tr['SeniorCitizen'].get('No', np.nan))} for others, {pct(tr['Partner'].get('Yes', np.nan))} with a partner vs {pct(tr['Partner'].get('No', np.nan))} without, and {pct(tr['AutoPay'].get('Yes', np.nan))} for automatic payers vs {pct(tr['AutoPay'].get('No', np.nan))} for others.

![Churn by contract and tenure](../images/02_churn_by_contract_and_tenure.png)
![Churn by internet service and payment method](../images/05_churn_by_internet_and_payment.png)
![Churn by add-ons](../images/06_churn_by_addons.png)

### 4.2 Model comparison (5-fold cross-validation on the training set)
{md_table(cv_show)}

![Model comparison](../images/09_model_comparison.png)

Best by mean ROC AUC: **{R['best_by_auc']}**. Models within one standard deviation of it: {', '.join(R['close'])}. The simplest of them, **{R['chosen']}**, was chosen and tuned (best parameters: {R['best_params']}; tuned cross-validated ROC AUC {R['tuned_auc']:.3f}).

### 4.3 Final model on the test set ({R['n_test']:,} customers, used once)
{md_table(test_show)}

Threshold chosen on training data: **{R['best_thr']:.2f}**.

- At the default threshold (0.50), the model catches **{d_tp} of {d_tp + d_fn}** churners and raises {d_fp} false alarms.
- At the chosen threshold ({R['best_thr']:.2f}), it catches **{tp} of {tp + fn}** churners ({pct(tp / (tp + fn))}) and raises {fp} false alarms, so about {pct(tp / (tp + fp) if tp + fp else 0)} of flagged customers really churn.
- ROC AUC on the test set is **{R['auc_test']:.3f}**.

![Confusion matrices](../images/11_confusion_matrices.png)
![ROC curve](../images/12_roc_curve.png)

### 4.4 What drives the predictions
Permutation importance on the test set (the drop in ROC AUC when a feature is shuffled):

{md_table(imp_tbl, "{:.4f}")}

![Permutation importance](../images/13_permutation_importance.png)

## 5. Insights and recommendations
{insights_md}

## 6. Limitations
- The data is a sample from a fictional company at a single point in time, so patterns may not transfer to a real business.
- The analysis shows association, not cause. Recommendations should be tested, for example with an A/B test, before wide rollout.
- Costs of outreach and the value of a retained customer were not provided, so the threshold was chosen by F1 rather than by profit.
- With {R['n_test']:,} test customers, small differences in the metrics are within normal noise.

## 7. Project structure and how to reproduce
```
README.md
requirements.txt
data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv   raw data (unchanged)
data/processed/telco_churn_cleaned.csv          final cleaned dataset
notebooks/end_to_end_churn_project.ipynb        complete workflow
docs/project_report.md                          this report
images/                                         charts
```
```bash
pip install -r requirements.txt
jupyter notebook notebooks/end_to_end_churn_project.ipynb   # Run All
```
"""
REPORT_PATH = DOCS_DIR / "project_report.md"
REPORT_PATH.write_text(report, encoding="utf-8")
print("Saved report to", REPORT_PATH)